## Implementazione di un Bot Discord per la Classificazione CIFAR-10
In questa sezione integriamo il modello addestrato con un Bot Discord. 
Il bot riceverà un'immagine, la processerà e restituirà la classe predetta.

In [1]:
# Installazione delle librerie necessarie
#!pip install discord.py tensorflow pillow nest-asyncio

import nest_asyncio
import io
import numpy as np
import tensorflow as tf
from PIL import Image
import discord
from discord.ext import commands

# Patch necessaria per far girare il loop di Discord dentro Jupyter
nest_asyncio.apply()  #Ho incluso nest_asyncio perché i file Jupyter hanno spesso conflitti con i loop di Discord.

##### 1. Caricamento del Modello e Definizione Classi
Carichiamo il file `.h5` generato durante la fase di training e definiamo le 10 etichette del dataset CIFAR-10.

In [2]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

MODEL_PATH = r'CODE\export_frontend\cifar10_improved_model.keras'


try:
    # Aggiungiamo compile=False per ignorare l'ottimizzatore PyTorch incompatibile
    model = tf.keras.models.load_model(MODEL_PATH, compile=False)
    print("✅ Modello caricato correttamente (modalità inferenza)!")
    
    # Visualizziamo la struttura per essere sicuri che sia integro
    model.summary()
    
except Exception as e:
    print(f"❌ Errore nel caricamento: {e}")

class_names = ['aereo', 'automobile', 'uccello', 'gatto', 'cervo', 
               'cane', 'rana', 'cavallo', 'nave', 'camion']

✅ Modello caricato correttamente (modalità inferenza)!


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ data_augmentation (Sequential)  │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 32, 32, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 16, 16, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 10)             │         1,29

 Total params: 404,778 (1.54 MB)

 Trainable params: 403,882 (1.54 MB)

 Non-trainable params: 896 (3.50 KB)

# Preprocessing

## 2. Funzione di Pre-processing
Le immagini inviate su Discord possono avere qualsiasi dimensione. La funzione `prepare_image` si occupa di:
1. Convertire l'immagine in RGB.
2. Ridimensionarla a 32x32 pixel.
3. Normalizzare i valori dei pixel tra 0 e 1.

In [3]:
'''from PIL import Image
def prepare_image(image_bytes):
    # .convert('RGB') elimina la trasparenza che confonde il modello
    img = Image.open(io.BytesIO(image_bytes)).convert('RGB')
    img = img.resize((32, 32))
    # Trasformiamo in float32 PRIMA della divisione per la massima precisione
    img_array = np.array(img).astype('float32') / 255.0
    return np.expand_dims(img_array, axis=0)'''

"from PIL import Image\ndef prepare_image(image_bytes):\n    # .convert('RGB') elimina la trasparenza che confonde il modello\n    img = Image.open(io.BytesIO(image_bytes)).convert('RGB')\n    img = img.resize((32, 32))\n    # Trasformiamo in float32 PRIMA della divisione per la massima precisione\n    img_array = np.array(img).astype('float32') / 255.0\n    return np.expand_dims(img_array, axis=0)"

In [4]:
'''def prepare_image(image_bytes):
    # Conversione in PNG prima del processamento (Standardizzazione del segnale)
    img_temp = Image.open(io.BytesIO(image_bytes))
    with io.BytesIO() as png_buffer:
        img_temp.save(png_buffer, format='PNG')
        png_buffer.seek(0)
        # Procediamo con l'immagine standardizzata in RGB
        img = Image.open(png_buffer).convert('RGB')
    
    img = img.resize((32, 32))
    # Normalizzazione: portiamo i pixel nel range [0, 1]
    img_array = np.array(img).astype('float32') / 255.0
    # Aggiungiamo la dimensione del batch (1, 32, 32, 3)
    return np.expand_dims(img_array, axis=0)'''

"def prepare_image(image_bytes):\n    # Conversione in PNG prima del processamento (Standardizzazione del segnale)\n    img_temp = Image.open(io.BytesIO(image_bytes))\n    with io.BytesIO() as png_buffer:\n        img_temp.save(png_buffer, format='PNG')\n        png_buffer.seek(0)\n        # Procediamo con l'immagine standardizzata in RGB\n        img = Image.open(png_buffer).convert('RGB')\n\n    img = img.resize((32, 32))\n    # Normalizzazione: portiamo i pixel nel range [0, 1]\n    img_array = np.array(img).astype('float32') / 255.0\n    # Aggiungiamo la dimensione del batch (1, 32, 32, 3)\n    return np.expand_dims(img_array, axis=0)"

In [5]:
def prepare_image(image_bytes):
    img = Image.open(io.BytesIO(image_bytes)).convert('RGB')
    img = img.resize((32, 32), Image.Resampling.LANCZOS)
    # Normalizzazione [0, 1] coerente con la cella 5 di CNN_base
    img_array = np.array(img).astype('float32') / 255.0
    return np.expand_dims(img_array, axis=0)

# Configurazione e Comandi Bot

### 3. Configurazione del Bot Discord
Definiamo il prefisso del comando (`!`) e la logica per gestire l'allegato. 
Il bot scaricherà l'immagine, userà il modello per la predizione e risponderà all'utente.

In [6]:
# Configurazione permessi (Intents)
intents = discord.Intents.default()
intents.message_content = True 
bot = commands.Bot(command_prefix="!", intents=intents)

In [7]:
'''import discord
from discord.ext import commands
import tensorflow as tf
import numpy as np
import io
from PIL import Image

# Configurazione permessi (Intents)
intents = discord.Intents.default()
intents.message_content = True 
bot = commands.Bot(command_prefix="!", intents=intents)


# 1. FUNZIONE DI SUPPORTO (Il Pre-processore)


# 2. RESET DEL SISTEMA (Per evitare errori in Jupyter)
#bot.remove_command('classifica')

# 3. IL COMANDO INTEGRATO (Cuore + Grafica)
@bot.command()
async def classifica(ctx):
    # Controllo preliminare: c'è un'immagine?
    if not ctx.message.attachments:
        await ctx.send("⚠️ Errore di input: Per favore, allega una foto al comando `!classifica`.")
        return

    attachment = ctx.message.attachments[0]
    
    # Filtro di frequenza (estensioni ammesse)
    if not any(attachment.filename.lower().endswith(ext) for ext in ['png', 'jpg', 'jpeg']):
        await ctx.send("❌ Formato non riconosciuto. Il sistema accetta solo JPG o PNG.")
        return

    # --- INIZIO FASE DINAMICA (Caricamento + Elaborazione) ---
    loading_msg = await ctx.send("⌛ [░░░░░░░░░░] 0% - Ricezione pacchetti dati...")
    
    try:
        # FASE A: Download e lettura (40%)
        image_bytes = await attachment.read()
        await loading_msg.edit(content="⌛ [████░░░░░░] 40% - Elaborazione segnale (Resize & Normalization)...")
        
        # FASE B: Preprocessing (70%)
        processed_img = prepare_image(image_bytes)
        await loading_msg.edit(content="⌛ [███████░░░] 70% - Interrogazione Rete Neurale CNN...")
        
        # FASE C: Inferenza (100%)
        # verbose=0 serve per non sporcare i log del server'''
'''preds = model.predict(processed_img, verbose=0)
        await loading_msg.edit(content="⌛ [██████████] 100% - Analisi completata!")
        
        # Calcolo statistico dei risultati
        index = np.argmax(preds)
        # Applichiamo Softmax per la distribuzione di probabilità
        probs = tf.nn.softmax(preds[0])
        accuracy = np.max(probs) * 100
        label = class_names[index]'''
        # Forza training=False per disattivare il Data Augmentation
'''preds = model(processed_img, training=False) 

        # Converti subito tutto in probabilità
        probs = tf.nn.softmax(preds[0]).numpy()
        index = np.argmax(probs)
        accuracy = probs[index] * 100
        label = class_names[index]

        # --- OUTPUT FINALE (Embed elegante) ---
        embed = discord.Embed(
            title="🔍 Risultato del Riconoscimento",
            description=f"Analisi completata per {ctx.author.mention}",
            color=0x3498db 
        )
        embed.set_thumbnail(url=attachment.url)
        embed.add_field(name="Oggetto Identificato", value=f"✨ **{label.upper()}**", inline=True)
        embed.add_field(name="Grado di Sicurezza", value=f"📈 {accuracy:.2f}%", inline=True)
        embed.set_footer(text="AI Model: CIFAR-10 CNN | MAGLGGICMADFMN")

        # Pulizia: rimuoviamo la barra di caricamento e inviamo il risultato finale
        await loading_msg.delete()
        await ctx.send(embed=embed)

    except Exception as e:
        await loading_msg.edit(content=f"❌ Errore critico di sistema: {e}")'''

'preds = model(processed_img, training=False) \n\n        # Converti subito tutto in probabilità\n        probs = tf.nn.softmax(preds[0]).numpy()\n        index = np.argmax(probs)\n        accuracy = probs[index] * 100\n        label = class_names[index]\n\n        # --- OUTPUT FINALE (Embed elegante) ---\n        embed = discord.Embed(\n            title="🔍 Risultato del Riconoscimento",\n            description=f"Analisi completata per {ctx.author.mention}",\n            color=0x3498db \n        )\n        embed.set_thumbnail(url=attachment.url)\n        embed.add_field(name="Oggetto Identificato", value=f"✨ **{label.upper()}**", inline=True)\n        embed.add_field(name="Grado di Sicurezza", value=f"📈 {accuracy:.2f}%", inline=True)\n        embed.set_footer(text="AI Model: CIFAR-10 CNN | MAGLGGICMADFMN")\n\n        # Pulizia: rimuoviamo la barra di caricamento e inviamo il risultato finale\n        await loading_msg.delete()\n        await ctx.send(embed=embed)\n\n    except Except

In [8]:
'''@bot.command()
async def classifica(ctx):
    if not ctx.message.attachments:
        await ctx.send("⚠️ Carica una foto e scrivi `!classifica`!")
        return

    attachment = ctx.message.attachments[0]
    loading_msg = await ctx.send("⌛ [░░░░░░░░░░] Analisi in corso...")
    
    try:
        image_bytes = await attachment.read()
        processed_img = prepare_image(image_bytes)
        print(f"DEBUG - Somma pixel: {np.sum(processed_img)}") 
        print(f"DEBUG - Primi 5 valori: {processed_img[0, 0, 0, :5]}")
        
        # 1. PREDIZIONE: Forza training=False per ignorare il Data Augmentation
        preds = model(processed_img, training=False)
        
        # 2. CALCOLO FISICO DELLE PROBABILITÀ (Softmax)
        probs = tf.nn.softmax(preds[0]).numpy()
        
        # 3. IDENTIFICAZIONE TOP 3
        # Prendiamo gli indici dei 3 valori più alti
        top_3_indices = np.argsort(probs)[-3:][::-1]
        
        # Costruiamo la stringa per l'Embed
        testo_top3 = ""
        for i in top_3_indices:
            nome_classe = class_names[i].capitalize()
            percentuale = probs[i] * 100
            testo_top3 += f"• **{nome_classe}**: {percentuale:.2f}%\n"

        # 4. CREAZIONE EMBED DETTAGLIATO
        index_primo = top_3_indices[0]
        label = class_names[index_primo]
        confidenza = probs[index_primo] * 100

        embed = discord.Embed(
            title="🔍 Analisi Rete Neurale (CIFAR-10)",
            description=f"Dati elaborati per {ctx.author.mention}",
            color=0x3498db
        )
        embed.set_thumbnail(url=attachment.url)
        embed.add_field(name="Risultato Principale", value=f"✨ **{label.upper()}**", inline=False)
        embed.add_field(name="Grado di Sicurezza", value=f"📈 {confidenza:.2f}%", inline=True)
        
        # QUESTA È LA PARTE CHE MANCAVA:
        embed.add_field(name="📊 Top 3 Probabilità", value=testo_top3, inline=False)
        
        embed.set_footer(text="Model: Improved CNN | Physics-based inference")

        await loading_msg.delete()
        await ctx.send(embed=embed)

    except Exception as e:
        await loading_msg.edit(content=f"❌ Errore tecnico: {e}")'''

'@bot.command()\nasync def classifica(ctx):\n    if not ctx.message.attachments:\n        await ctx.send("⚠️ Carica una foto e scrivi `!classifica`!")\n        return\n\n    attachment = ctx.message.attachments[0]\n    loading_msg = await ctx.send("⌛ [░░░░░░░░░░] Analisi in corso...")\n\n    try:\n        image_bytes = await attachment.read()\n        processed_img = prepare_image(image_bytes)\n        print(f"DEBUG - Somma pixel: {np.sum(processed_img)}") \n        print(f"DEBUG - Primi 5 valori: {processed_img[0, 0, 0, :5]}")\n\n        # 1. PREDIZIONE: Forza training=False per ignorare il Data Augmentation\n        preds = model(processed_img, training=False)\n\n        # 2. CALCOLO FISICO DELLE PROBABILITÀ (Softmax)\n        probs = tf.nn.softmax(preds[0]).numpy()\n\n        # 3. IDENTIFICAZIONE TOP 3\n        # Prendiamo gli indici dei 3 valori più alti\n        top_3_indices = np.argsort(probs)[-3:][::-1]\n\n        # Costruiamo la stringa per l\'Embed\n        testo_top3 = ""\n 

In [9]:
'''@bot.command()
async def classifica(ctx):
    if not ctx.message.attachments:
        await ctx.send("⚠️ Carica una foto!")
        return

    attachment = ctx.message.attachments[0]
    
    # Messaggio di attesa per l'utente
    loading_msg = await ctx.send("⌛ [░░░░░░░░░░] Analisi in corso...")
    
    try:
        # 1. ACQUISIZIONE E PRE-ELABORAZIONE
        image_bytes = await attachment.read()
        processed_img = prepare_image(image_bytes)
        
        # Calcolo diagnostico del segnale
        pixel_sum = np.sum(processed_img)
        
        # 2. INFERENZA
        # training=False è fondamentale perché il tuo modello ha layer di augmentation
        preds = model(processed_img, training=False)
        
        # 3. ELABORAZIONE RISULTATI
        # Il tuo modello ha già 'softmax' come activation nell'ultimo layer
        # Prendiamo direttamente i valori predetti (che sono già probabilità)
        probs = preds[0].numpy() 
        
        # Identifichiamo i 3 risultati migliori
        top_3_indices = np.argsort(probs)[-3:][::-1]
        
        testo_top3 = ""
        for i in top_3_indices:
            nome = class_names[i].upper()
            percentuale = probs[i] * 100
            testo_top3 += f"• **{nome}**: {percentuale:.2f}%\n"

        # 4. CREAZIONE DELL'EMBED FINALE
        index_best = top_3_indices[0]
        label_best = class_names[index_best]
        confidenza_best = probs[index_best] * 100

        embed = discord.Embed(
            title="🔍 Analisi Tecnica Rete Neurale",
            description=f"Risultati per {ctx.author.mention}",
            color=0x3498db
        )
        embed.set_thumbnail(url=attachment.url)
        embed.add_field(name="Predizione Principale", value=f"✨ **{label_best.upper()}**", inline=False)
        embed.add_field(name="Grado di Sicurezza", value=f"📈 {confidenza_best:.2f}%", inline=True)
        embed.add_field(name="📊 Distribuzione Probabilità", value=testo_top3, inline=False)
        embed.set_footer(text=f"Check sum: {pixel_sum:.2f} | Info: Softmax integrata")

        # Rimuoviamo il caricamento e inviamo il risultato
        await loading_msg.delete()
        await ctx.send(embed=embed)

    except Exception as e:
        await loading_msg.edit(content=f"❌ Errore durante l'analisi: {e}")'''

'@bot.command()\nasync def classifica(ctx):\n    if not ctx.message.attachments:\n        await ctx.send("⚠️ Carica una foto!")\n        return\n\n    attachment = ctx.message.attachments[0]\n\n    # Messaggio di attesa per l\'utente\n    loading_msg = await ctx.send("⌛ [░░░░░░░░░░] Analisi in corso...")\n\n    try:\n        # 1. ACQUISIZIONE E PRE-ELABORAZIONE\n        image_bytes = await attachment.read()\n        processed_img = prepare_image(image_bytes)\n\n        # Calcolo diagnostico del segnale\n        pixel_sum = np.sum(processed_img)\n\n        # 2. INFERENZA\n        # training=False è fondamentale perché il tuo modello ha layer di augmentation\n        preds = model(processed_img, training=False)\n\n        # 3. ELABORAZIONE RISULTATI\n        # Il tuo modello ha già \'softmax\' come activation nell\'ultimo layer\n        # Prendiamo direttamente i valori predetti (che sono già probabilità)\n        probs = preds[0].numpy() \n\n        # Identifichiamo i 3 risultati miglio

In [ ]:
# Definizione delle classi nell'ordine esatto del dataset CIFAR-10
class_names = [
    'aereo',        # indice 0
    'automobile',   # indice 1
    'uccello',      # indice 2
    'gatto',        # indice 3
    'cervo',        # indice 4
    'cane',         # indice 5
    'rana',         # indice 6
    'cavallo',      # indice 7
    'nave',         # indice 8
    'camion'        # indice 9
]

In [1]:
@bot.command()
async def classifica(ctx):
    if not ctx.message.attachments:
        await ctx.send("⚠️ Allega una foto!")
        return

    attachment = ctx.message.attachments[0]
    # Messaggio di caricamento per feedback immediato
    loading_msg = await ctx.send("⌛ Analisi in corso...")
    
    try:
        image_bytes = await attachment.read()
        processed_img = prepare_image(image_bytes)
        
        # INFERENZA: training=False disattiva RandomRotation/Flip del modello
        preds = model(processed_img, training=False)
        
        # Il modello restituisce probabilità (Softmax è già presente)
        probs = preds[0].numpy()
        index = np.argmax(probs)
        confidenza = probs[index] * 100
        
        # Creazione dell'Embed
        embed = discord.Embed(title="🔍 Analisi CIFAR-10", color=0x3498db)
        embed.set_thumbnail(url=attachment.url)
        embed.add_field(name="Oggetto Identificato", value=f"✨ **{class_names[index].upper()}**", inline=False)
        embed.add_field(name="Grado di Sicurezza", value=f"📈 {confidenza:.2f}%", inline=True)
        
        # Diagnostica per il Check-sum dei pixel (deve variare tra immagini diverse)
        p_sum = np.sum(processed_img)
        embed.set_footer(text=f"Check-sum: {p_sum:.2f} | Mode: Inference")

        await loading_msg.delete()
        await ctx.send(embed=embed)

    except Exception as e:
        await ctx.send(f"❌ Errore critico: {e}")

NameError: name 'bot' is not defined

### Stato Personalizzato
bot mostra cosa sta facendo sotto il suo nome nella lista utenti.

In [11]:
@bot.event
async def on_ready():
    # Imposta lo stato: "In ascolto di !classifica"
    activity = discord.Activity(type=discord.ActivityType.listening, name="!classifica")
    await bot.change_presence(status=discord.Status.online, activity=activity)
    print(f'✅ Bot pronto: {bot.user}')

### Gestione Errori

In [12]:
# 1. Configurazione Stato Personalizzato
@bot.event
async def on_ready():
    # Imposta lo stato: "In ascolto di !info"
    activity = discord.Activity(type=discord.ActivityType.listening, name="!info")
    await bot.change_presence(status=discord.Status.online, activity=activity)
    print(f'✅ Bot online come: {bot.user}')

# 2. Gestione Errori Globale
@bot.event
async def on_command_error(ctx, error):
    if isinstance(error, commands.CommandNotFound):
        await ctx.send("❓ Comando non riconosciuto. Scrivi `!info` per vedere cosa posso fare.")
    elif isinstance(error, commands.MissingRequiredArgument):
        await ctx.send("⚠️ Mancano degli argomenti nel comando.")
    else:
        print(f"Errore non gestito: {error}")

### Comando Classifica con Barra

In [13]:
'''@bot.command()
async def classifica(ctx):
    # Controllo allegati
    if not ctx.message.attachments:
        await ctx.send("⚠️ Per favore, carica un'immagine insieme al comando `!classifica`.")
        return

    attachment = ctx.message.attachments[0]
    
    # Filtro estensioni
    if not any(attachment.filename.lower().endswith(ext) for ext in ['png', 'jpg', 'jpeg']):
        await ctx.send("❌ Formato file non supportato. Usa solo JPG o PNG.")
        return

    # Inizio caricamento grafico
    loading_msg = await ctx.send("⌛ [░░░░░░░░░░] 0% - Ricezione immagine...")
    
    try:
        # Download (40%)
        image_bytes = await attachment.read()
        await loading_msg.edit(content="⌛ [████░░░░░░] 40% - Elaborazione pixel...")
        
        # Preprocessing (70%)
        processed_img = prepare_image(image_bytes)
        await loading_msg.edit(content="⌛ [███████░░░] 70% - Interrogazione Rete Neurale...")
        
        # Predizione (100%)
        preds = model.predict(processed_img)
        await loading_msg.edit(content="⌛ [██████████] 100% - Analisi completata!")
        
        # Calcolo risultati
        index = np.argmax(preds)
        accuracy = np.max(tf.nn.softmax(preds[0])) * 100
        label = class_names[index]

        # Creazione dell'Embed finale
        embed = discord.Embed(
            title="🔍 Risultato del Riconoscimento",
            description=f"Analisi completata per l'utente {ctx.author.mention}",
            color=0x3498db # Colore blu elegante
        )
        embed.set_thumbnail(url=attachment.url)
        embed.add_field(name="Oggetto Predetto", value=f"**{label.upper()}**", inline=True)
        embed.add_field(name="Grado di Sicurezza", value=f"{accuracy:.2f}%", inline=True)
        embed.set_footer(text="AI Model: CIFAR-10 CNN | UniBari Project")

        # Rimuove la barra e invia il risultato
        await loading_msg.delete()
        await ctx.send(embed=embed)

    except Exception as e:
        await loading_msg.edit(content=f"❌ Si è verificato un errore durante l'analisi: {e}")'''

'@bot.command()\nasync def classifica(ctx):\n    # Controllo allegati\n    if not ctx.message.attachments:\n        await ctx.send("⚠️ Per favore, carica un\'immagine insieme al comando `!classifica`.")\n        return\n\n    attachment = ctx.message.attachments[0]\n\n    # Filtro estensioni\n    if not any(attachment.filename.lower().endswith(ext) for ext in [\'png\', \'jpg\', \'jpeg\']):\n        await ctx.send("❌ Formato file non supportato. Usa solo JPG o PNG.")\n        return\n\n    # Inizio caricamento grafico\n    loading_msg = await ctx.send("⌛ [░░░░░░░░░░] 0% - Ricezione immagine...")\n\n    try:\n        # Download (40%)\n        image_bytes = await attachment.read()\n        await loading_msg.edit(content="⌛ [████░░░░░░] 40% - Elaborazione pixel...")\n\n        # Preprocessing (70%)\n        processed_img = prepare_image(image_bytes)\n        await loading_msg.edit(content="⌛ [███████░░░] 70% - Interrogazione Rete Neurale...")\n\n        # Predizione (100%)\n        preds =

### Comando Informativo
Aggiungiamo un comando per spiegare agli utenti quali sono le 10 categorie che il modello può riconoscere (dataset CIFAR-10) e come utilizzare il bot.

In [14]:
@bot.command()
async def info(ctx):
    # Creiamo un messaggio formattato in modo elegante (Embed)
    descrizione = (
        "Ciao! Sono il Bot del progetto di Classificazione Immagini.\n"
        "Utilizzo una rete neurale (CNN) addestrata sul dataset **CIFAR-10**.\n\n"
        "**Cosa posso fare?**\n"
        "Se mi invii una foto e scrivi `!classifica`, proverò a capire cosa rappresenta.\n\n"
        "**Le mie 10 categorie sono:**\n"
        f"✈️ {class_names[0]}, 🚗 {class_names[1]}, 🐦 {class_names[2]}, 🐱 {class_names[3]}, 🦌 {class_names[4]},\n"
        f"🐶 {class_names[5]}, 🐸 {class_names[6]}, 🐴 {class_names[7]}, 🚢 {class_names[8]}, 🚛 {class_names[9]}.\n\n"
        "**Istruzioni:**\n"
        "Carica una foto e scrivi `!classifica` nel campo del commento!"
    )
    
    await ctx.send(descrizione)

# Esecuzione
Inserisci il tuo Token e avvia la cella. Il bot rimarrà attivo finché la cella è in esecuzione.

In [ ]:
TOKEN = 'MTQ4MDU1NzkxMjc5NDMzNzMzMA.G0pjlc.3iao-ooRdYTqBvBSn1j579jY8nFKp3aDMpBrmI'
bot.run(TOKEN)

[2026-03-09 19:06:47] [INFO    ] discord.client: logging in using static token
[2026-03-09 19:06:49] [INFO    ] discord.gateway: Shard ID None has connected to Gateway (Session ID: 1274250a1329885fe26a0f227a6ff4b3).


✅ Bot online come: image recognizer bot#6774


### Come ottenere il Token (Passaggi Fondamentali)
Prima di eseguire il codice, devi registrare il bot:

- Vai su Discord Developer Portal.

- Clicca su "New Application" e dai un nome.

- Vai a sinistra su "Bot".

- Clicca su "Reset Token" per visualizzare e copiare il tuo Token.

IMPORTANTE: Scorri verso il basso nella pagina Bot e attiva "Message Content Intent". Senza questo, il bot non leggerà il comando !classifica.

### Caricamento nella cartella del progetto
Una volta che hai verificato che tutto funziona localmente:

Puoi caricare il file .ipynb nella cartella del progetto su GitHub o Drive.

Ricordati di non caricare mai il Token in chiaro su GitHub pubblico (usa un file .env o cancellalo prima di caricare).